# Personnel Stress & Welfare Risk Prediction
Predicting `risk_band` (Low / Medium / High / Critical) from HR, deployment,
duty, self-assessment, biometric, and engagement features.

**Pipeline:** Setup → Load → Train/Test Split → EDA → Preprocessing →
Feature Extraction → Encoding → Scaling → Model Training & Hyperparameter Tuning → Evaluation

> Upload `personnel_stress_welfare_synthetic_dataset.csv` to the Colab session
> (or mount Drive) before running.

## 1. Setup

In [ ]:
!pip install -q xgboost imbalanced-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                              ConfusionMatrixDisplay, balanced_accuracy_score)

from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 100)

## 2. Load Data

In [ ]:
# If running in Colab, upload the CSV first:
# from google.colab import files
# uploaded = files.upload()

DATA_PATH = "personnel_stress_welfare_synthetic_dataset.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 3. Define Target & Drop Leakage / Identifier Columns

`risk_score`, `risk_trend_direction`, `model_confidence_score`,
`top_contributing_factor_categories`, and every `intervention_*` column are
**downstream outputs derived from `risk_band`** — they must be excluded from
the feature set or the model will trivially "predict" the label from itself.

`pseudo_id` is a unique identifier (no predictive value). `unit_id` is a
high-cardinality identifier whose informative content is already captured by
the unit-level aggregate columns (`unit_strength_vs_sanctioned_ratio`,
`unit_attrition_rate_qtr`, `unit_incident_count_qtr`), so it is dropped too.

In [ ]:
TARGET = "risk_band"

LEAKAGE_COLS = [
    "risk_score", "risk_trend_direction", "model_confidence_score",
    "top_contributing_factor_categories", "intervention_type",
    "intervention_status", "personnel_acceptance_flag",
    "follow_up_scheduled_flag", "intervention_outcome",
]
ID_COLS = ["pseudo_id", "unit_id"]

X = df.drop(columns=LEAKAGE_COLS + ID_COLS + [TARGET])
y = df[TARGET]

print("Feature matrix:", X.shape)
y.value_counts()

## 4. Train/Test Split
Split **before** any EDA-driven decisions or preprocessing are fit, so the
test set stays a genuinely unseen holdout. Stratify on the target since the
risk bands are imbalanced (mirrors the real-world minority of high-risk
personnel).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("\nTrain class balance:\n", y_train.value_counts(normalize=True).round(3))
print("\nTest class balance:\n", y_test.value_counts(normalize=True).round(3))

## 5. Exploratory Data Analysis (EDA)
All EDA below runs on `X_train` / `y_train` only — never touch the test set
during exploration, to avoid information leaking into modeling decisions.

In [ ]:
train_df = X_train.copy()
train_df["risk_band"] = y_train.values

# --- Target distribution ---
order = ["Low","Medium","High","Critical"]
plt.figure(figsize=(6,4))
sns.countplot(data=train_df, x="risk_band", order=order, hue="risk_band", palette="viridis", legend=False)
plt.title("Risk Band Distribution (Train Set)")
plt.show()

In [ ]:
# --- Missingness map ---
missing_pct = train_df.isna().mean().sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

plt.figure(figsize=(8,5))
sns.barplot(x=missing_pct.values, y=missing_pct.index, hue=missing_pct.index, palette="rocket", legend=False)
plt.xlabel("Fraction Missing")
plt.title("Missing Values by Column (Train Set)")
plt.show()

print(missing_pct)

In [ ]:
# --- Numeric feature distributions vs risk band ---
key_numeric = ["weekly_hours_avg_rolling_4wk", "duty_hours_variability",
               "hardship_rating", "family_separation_days_cumulative",
               "job_satisfaction_score", "helpline_access_count_last_year"]

fig, axes = plt.subplots(2, 3, figsize=(16,9))
for ax, col in zip(axes.ravel(), key_numeric):
    sns.boxplot(data=train_df, x="risk_band", y=col, order=order, ax=ax, hue="risk_band", palette="viridis", legend=False)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation heatmap (numeric features only) ---
numeric_cols_preview = X_train.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(16,12))
corr = train_df[numeric_cols_preview].corr()
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.3)
plt.title("Numeric Feature Correlation Heatmap")
plt.show()

In [ ]:
# --- Categorical breakdown: risk band by force type & role category ---
fig, axes = plt.subplots(1, 2, figsize=(16,5))
sns.countplot(data=train_df, x="force_type", hue="risk_band", hue_order=order, ax=axes[0])
axes[0].tick_params(axis="x", rotation=45)
axes[0].set_title("Risk Band by Force Type")

sns.countplot(data=train_df, x="role_category", hue="risk_band", hue_order=order, ax=axes[1])
axes[1].tick_params(axis="x", rotation=30)
axes[1].set_title("Risk Band by Role Category")
plt.tight_layout()
plt.show()

## 6. Feature Extraction (Derived Features)
Engineer a few composite indices that summarize related raw columns into a
single interpretable signal — useful both for model performance and for the
"explainable risk factor" requirement in the original problem statement.

These are computed with `.fit`/`.transform`-style functions on **train first**,
then applied identically to test, so no test-set statistics leak into train.

In [ ]:
def engineer_features(frame):
    f = frame.copy()

    # Workload strain index: overtime load + schedule irregularity
    f["workload_strain_index"] = (
        (f["weekly_hours_avg_rolling_4wk"] - 48).clip(lower=0) * 0.5
        + f["duty_hours_variability"] * 0.5
    )

    # Deployment burden index: hardship + family separation, scaled
    f["deployment_burden_index"] = (
        f["hardship_rating"] * 10
        + f["family_separation_days_cumulative"] / 30
        + f["consecutive_deployments_flag"] * 15
        + f["combat_exposure_flag"] * 10
    )

    # Leave irregularity index
    f["leave_irregularity_index"] = (
        f["leave_denied_count"] * 2 + f["emergency_leave_count_last_year"] * 1.5
    )

    # Career instability index
    f["career_instability_index"] = (
        f["transfer_within_probation_flag"] * 10
        + f["hardship_posting_consecutive_count"] * 3
        + (1 / f["avg_tenure_per_posting_years"].clip(lower=0.3))
    )

    # Engagement decline signal (only meaningful for consenting personnel)
    f["engagement_decline_signal"] = (
        f["withdrawal_flag"] * 10
        - f["app_login_frequency_weekly"]
        - f["self_assessment_completion_rate"] * 5
    )

    # Unit strain index
    f["unit_strain_index"] = (
        f["unit_attrition_rate_qtr"] + (1 - f["unit_strength_vs_sanctioned_ratio"]) * 20
        + f["unit_incident_count_qtr"] * 2
    )

    return f

X_train_fe = engineer_features(X_train)
X_test_fe = engineer_features(X_test)

new_feats = ["workload_strain_index","deployment_burden_index","leave_irregularity_index",
             "career_instability_index","engagement_decline_signal","unit_strain_index"]
X_train_fe[new_feats].describe().T

## 7. Feature Encoding & Scaling Setup
- **Ordinal** columns (naturally ordered bands) → `OrdinalEncoder` with explicit category order.
- **Nominal** categorical columns → `OneHotEncoder`.
- **Numeric** columns → median imputation (with a missing-indicator, since
  missingness itself is informative — e.g. non-consenting personnel) + `StandardScaler`.

All of this is wrapped in a single `ColumnTransformer` inside a `Pipeline`,
fit only on the training data, so preprocessing parameters (medians, scale
factors, one-hot categories) never leak from the test set.

In [ ]:
ordinal_specs = {
    "age_band": ["21-25","26-30","31-35","36-40","41-45","46-50","51-55"],
    "years_of_service_band": ["0-2","3-5","6-10","11-15","16-20","21-25","26+"],
    "dependents_count_band": ["0","1-2","3-4","5+"],
    "distance_from_home_band": ["<100km","100-500km","500-1500km",">1500km"],
}
ordinal_cols = list(ordinal_specs.keys())

nominal_cols = ["force_type","rank_band","role_category","gender","marital_status",
                "medical_category","education_level","posting_type_current","home_region_band"]

numeric_cols = [c for c in X_train_fe.columns if c not in ordinal_cols + nominal_cols]

print(f"{len(numeric_cols)} numeric | {len(nominal_cols)} nominal | {len(ordinal_cols)} ordinal")

In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])

ordinal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(categories=[ordinal_specs[c] for c in ordinal_cols])),
])

nominal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
    ("ord", ordinal_pipe, ordinal_cols),
    ("nom", nominal_pipe, nominal_cols),
])

In [ ]:
# Encode the ordinal target (Low < Medium < High < Critical)
le = LabelEncoder()
le.fit(["Low","Medium","High","Critical"])

y_train_enc = le.transform(y_train)
y_test_enc = le.transform(y_test)

print(dict(zip(le.classes_, le.transform(le.classes_))))

## 8. Model Training & Hyperparameter Tuning
Compare three model families inside the same `Pipeline` (preprocessor + model),
each tuned with `RandomizedSearchCV` under stratified 5-fold CV, scored on
**macro F1** (treats all four risk bands equally — important since we care
about catching the minority High/Critical cases, not just overall accuracy).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING = "f1_macro"

search_results = {}

In [ ]:
# --- Model 1: Logistic Regression (baseline, linear) ---
logreg_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

logreg_param_dist = {
    "model__C": [0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10],
    "model__class_weight": [None, "balanced"],
}

logreg_search = RandomizedSearchCV(
    logreg_pipe, logreg_param_dist, n_iter=10, scoring=SCORING,
    cv=cv, random_state=RANDOM_STATE, n_jobs=-1, verbose=1,
)
logreg_search.fit(X_train_fe, y_train_enc)
search_results["LogisticRegression"] = logreg_search
print("Best CV macro-F1:", logreg_search.best_score_, "\nBest params:", logreg_search.best_params_)

In [ ]:
# --- Model 2: Random Forest ---
rf_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE)),
])

rf_param_dist = {
    "model__n_estimators": [200, 300, 500, 700],
    "model__max_depth": [None, 6, 10, 14, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None],
    "model__class_weight": [None, "balanced", "balanced_subsample"],
}

rf_search = RandomizedSearchCV(
    rf_pipe, rf_param_dist, n_iter=20, scoring=SCORING,
    cv=cv, random_state=RANDOM_STATE, n_jobs=-1, verbose=1,
)
rf_search.fit(X_train_fe, y_train_enc)
search_results["RandomForest"] = rf_search
print("Best CV macro-F1:", rf_search.best_score_, "\nBest params:", rf_search.best_params_)

In [ ]:
# --- Model 3: XGBoost ---
xgb_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", XGBClassifier(
        objective="multi:softmax", num_class=4, eval_metric="mlogloss",
        random_state=RANDOM_STATE, use_label_encoder=False,
    )),
])

xgb_param_dist = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__min_child_weight": [1, 3, 5],
}

xgb_search = RandomizedSearchCV(
    xgb_pipe, xgb_param_dist, n_iter=20, scoring=SCORING,
    cv=cv, random_state=RANDOM_STATE, n_jobs=-1, verbose=1,
)
xgb_search.fit(X_train_fe, y_train_enc)
search_results["XGBoost"] = xgb_search
print("Best CV macro-F1:", xgb_search.best_score_, "\nBest params:", xgb_search.best_params_)

## 9. Model Comparison & Selection

In [ ]:
comparison = pd.DataFrame({
    "Model": list(search_results.keys()),
    "Best CV Macro-F1": [s.best_score_ for s in search_results.values()],
})
comparison = comparison.sort_values("Best CV Macro-F1", ascending=False).reset_index(drop=True)
print(comparison)

best_model_name = comparison.loc[0, "Model"]
best_search = search_results[best_model_name]
best_estimator = best_search.best_estimator_
print(f"\nSelected best model: {best_model_name}")

## 10. Final Evaluation on Held-Out Test Set
The test set has not been touched until now — this is the only honest
estimate of how the model would perform on new personnel records.

In [ ]:
y_pred = best_estimator.predict(X_test_fe)

print(classification_report(y_test_enc, y_pred, target_names=le.classes_))
print("Balanced accuracy:", round(balanced_accuracy_score(y_test_enc, y_pred), 3))
print("Macro F1:", round(f1_score(y_test_enc, y_pred, average="macro"), 3))

In [ ]:
cm = confusion_matrix(y_test_enc, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
fig, ax = plt.subplots(figsize=(6,6))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title(f"Confusion Matrix — {best_model_name} (Test Set)")
plt.show()

In [ ]:
# --- Feature importance (tree-based models only) ---
if best_model_name in ("RandomForest", "XGBoost"):
    ohe_cols = best_estimator.named_steps["prep"].named_transformers_["nom"]\
        .named_steps["encoder"].get_feature_names_out(nominal_cols)
    num_indicator_cols = best_estimator.named_steps["prep"].named_transformers_["num"]\
        .named_steps["imputer"].get_feature_names_out(numeric_cols)
    all_feature_names = list(num_indicator_cols) + ordinal_cols + list(ohe_cols)

    importances = best_estimator.named_steps["model"].feature_importances_
    fi = pd.Series(importances, index=all_feature_names).sort_values(ascending=False).head(20)

    plt.figure(figsize=(8,8))
    sns.barplot(x=fi.values, y=fi.index, hue=fi.index, palette="viridis", legend=False)
    plt.title(f"Top 20 Feature Importances — {best_model_name}")
    plt.tight_layout()
    plt.show()
else:
    print("Selected model has no native feature_importances_ (Logistic Regression) — "
          "inspect model.coef_ instead if needed.")

## 11. Save the Final Model

In [ ]:
import joblib

joblib.dump(best_estimator, "personnel_risk_best_model.joblib")
joblib.dump(le, "risk_band_label_encoder.joblib")
print("Saved: personnel_risk_best_model.joblib, risk_band_label_encoder.joblib")